<a href="https://colab.research.google.com/github/hibbo67/ai-study-assistant/blob/main/algorithms_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project: The Grokking Algorithms Assistant

## Problem
I have a PDF of "Grokking Algorithms" with hundreds of pages. I want to ask specific questions about the book (like "What is Big O?" or "How does quicksort work?") and get answers grounded strictly in the text, complete with the page number so I can read more.

## Users
Me, primarily. Later, anyone studying algorithms.

## Success looks like
- It answers questions accurately using the book's text.
- It refuses to answer questions not in the book (e.g., "What is the capital of France?").
- It tells me which page the answer came from.

## My 10 Test Questions (The Golden Set)
1. What is the run time for binary search? → Log time / O(log n)
2. How does selection sort work? → It takes O(n²) time, finds smallest and appends to new list.
3. What is the difference between an array and a linked list? → Arrays have fast reads, linked lists have fast inserts.
4. How does recursion work? → Base case and recursive case.
5. What is the worst-case run time of quicksort? → O(n²)
6. What is a hash table used for? → Lookups, caching, preventing duplicates.
7. What is a directed graph? → A graph where edges have arrows.
8. What is the Dijkstra's algorithm used for? → Finding the shortest path in a weighted graph.
9. What is a greedy algorithm? → At each step, pick the optimal move (locally optimal).
10. What is the knapsack problem? → Maximizing value given a weight constraint.

In [1]:
!pip install -q chromadb google-genai pypdf
from google.colab import userdata, files
from google import genai
from chromadb import Documents, EmbeddingFunction, Embeddings
import chromadb
import time
import re
import os

# 1. Setup Clients
api_keys = [userdata.get(f"GEMINI_KEY_{i}") for i in range(1, 8)]
ALL_KEYS = [k for k in api_keys if k]

# Initialize a client for each available key
import google.genai.errors as genai_errors
embed_clients = [genai.Client(api_key=key) for key in ALL_KEYS]

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, clients):
        self.clients = clients
        self.current_client_idx = 0

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            success = False
            retries = 0
            # Try up to 10 times per text chunk
            while not success and retries < 10:
                client = self.clients[self.current_client_idx]
                try:
                    result = client.models.embed_content(
                        model="gemini-embedding-2",
                        contents=[text],
                    )
                    vectors.append([float(x) for x in result.embeddings[0].values])
                    success = True
                except Exception as e:
                    if "429" in str(e):
                        # Cycle to the next key on 429 rate limit
                        self.current_client_idx = (self.current_client_idx + 1) % len(self.clients)
                        retries += 1
                        print(f"Rate limit hit. Rotating to key {self.current_client_idx + 1} and waiting 4 seconds...")
                        time.sleep(4)
                    else:
                        raise e
        return vectors

# 2. Automated File Selection
pdf_candidates = ["algorithms.PDF", "algorithms.pdf", "algorithms (1).PDF", "algorithms (1).pdf"]
pdf_filename = None

for candidate in pdf_candidates:
    if os.path.exists(candidate):
        pdf_filename = candidate
        break

if not pdf_filename:
    print("No local PDF found. Please upload your algorithms.pdf file:")
    uploaded = files.upload()
    pdf_filename = list(uploaded.keys())[0]
else:
    print(f"Found existing PDF in workspace: {pdf_filename}")

# 3. Extract & Chunk (Paragraph-based chunking to preserve quota!)
from pypdf import PdfReader
reader = PdfReader(pdf_filename)

all_chunks = []
all_metadatas = []
all_ids = []

for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if not text:
        continue

    # Split by double newline to get coherent paragraphs instead of single lines
    page_chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]

    for chunk in page_chunks:
        all_chunks.append(chunk)
        all_metadatas.append({"page": page_num + 1})
        all_ids.append(f"page{page_num+1}_chunk{len(all_chunks)}")

print(f"Extracted {len(all_chunks)} paragraph chunks from {len(reader.pages)} pages.")

# 4. Store in Chroma
db = chromadb.Client()
try:
    db.delete_collection("algorithms_book")
except:
    pass
col = db.create_collection(name="algorithms_book", embedding_function=GeminiEmbeddingFunction(embed_clients))

batch_size = 50
for i in range(0, len(all_chunks), batch_size):
    success = False
    while not success:
        try:
            col.add(
                ids=all_ids[i:i+batch_size],
                documents=all_chunks[i:i+batch_size],
                metadatas=all_metadatas[i:i+batch_size]
            )
            success = True
            print(f"Ingested {min(i+batch_size, len(all_chunks))} / {len(all_chunks)}...")
        except Exception as e:
            if "429" in str(e):
                print("⚠️ All keys rate-limited. Waiting 25 seconds for quota reset...")
                time.sleep(25)
            else:
                raise e

print("✅ Book fully ingested!")

Found existing PDF in workspace: algorithms.PDF
Extracted 310 paragraph chunks from 322 pages.
Ingested 50 / 310...
Ingested 100 / 310...
Rate limit hit. Rotating to key 2 and waiting 4 seconds...
Ingested 150 / 310...
Ingested 200 / 310...
Rate limit hit. Rotating to key 3 and waiting 4 seconds...
Ingested 250 / 310...
Ingested 300 / 310...
Rate limit hit. Rotating to key 4 and waiting 4 seconds...
Ingested 310 / 310...
✅ Book fully ingested!


```markdown
## Ingesting the Full PDF
Run the cell below to upload your "Grokking Algorithms" PDF, extract its contents page-by-page, chunk the text, and load it into your Chroma collection.
```

In [12]:
# Optimized Golden Set Evaluation (Quota-Friendly Batch Judge)
import time
import json
import re
import sys
from google import genai

# 1. Self-contained RAG Pipeline Definition with Robust 503/429 Handling
def ask_book(question):
    hits = col.query(query_texts=[question], n_results=4)
    retrieved_docs = hits["documents"][0]
    distances = hits["distances"][0]

    if distances[0] > 0.85:
        return "I don't know. That information isn't in the book."

    context = "\n\n".join(retrieved_docs)
    prompt = f"""Answer the user's question using ONLY the context below.
If the context does not contain the answer, you must say "I don't know."
Cite your sources using the bracketed numbers [1], [2], etc. corresponding to context chunks.

Context:
{context}

Question: {question}
"""

    for key in ALL_KEYS:
        for attempt in range(2):
            try:
                temp_client = genai.Client(api_key=key)
                response = temp_client.models.generate_content(
                    model="gemini-3.8-flash",
                    contents=prompt,
                )
                return response.text
            except Exception as e:
                err_msg = str(e).lower()
                if "429" in err_msg or "503" in err_msg:
                    print(f"⚠️ Service spike or 429 hit. Retrying with next available key...", flush=True)
                    time.sleep(4)
                    break
                else:
                    raise e
    return "I don't know. That information isn't in the book."

# Security/Guardrail logic
BAD_PATTERNS = ["ignore previous", "ignore all", "reveal your system", "you are now", "system prompt"]

def safe_ask_book(question):
    for pattern in BAD_PATTERNS:
        if re.search(pattern, question.lower()):
            return "I don't know. That information isn't in the book."
    return ask_book(question)

# 2. Golden Set
golden_set = [
    {"q": "What is the run time for binary search?", "expected": "O(log n)"},
    {"q": "How does selection sort work?", "expected": "finds the smallest item, adds to new list"},
    {"q": "What is the difference between an array and a linked list?", "expected": "arrays = fast reads, lists = fast inserts"},
    {"q": "What is a recursive function?", "expected": "calls itself, base case"},
    {"q": "What is the worst-case run time of quicksort?", "expected": "O(n^2)"},
    {"q": "What is a hash table used for?", "expected": "lookups, caching, preventing duplicates"},
    {"q": "What is a directed graph?", "expected": "graph with arrows (one-way relationships)"},
    {"q": "What is Dijkstra's algorithm used for?", "expected": "shortest path in weighted graph"},
    {"q": "What is a greedy algorithm?", "expected": "locally optimal move"},
    {"q": "What is the knapsack problem?", "expected": "maximizing value given a weight constraint"},
]

BATCH_RUBRIC = """You are an expert evaluator scoring a study assistant.
You will be given a list of test cases, each containing a Question, the Expected Answer, and the AI's Actual Answer.
Return a JSON object with a single key "results", containing a list of objects:
{
  "results": [
     {
       "q_index": <int>,
       "score": <int 1-5>,
       "reason": "<short sentence>"
     }
  ]
}
Score 5 if the AI's answer contains the core truth of the expected answer.
Score 1 if the AI's answer is "I don't know" or totally wrong.
"""

def judge_all_answers(test_runs):
    prompt_data = []
    for idx, run in enumerate(test_runs):
        prompt_data.append(f"""Index: {idx}
Question: {run['q']}
Expected Answer: {run['expected']}
AI's Answer: {run['actual']}
---""")
    prompt = "\n".join(prompt_data) + "\nScore all the above inputs according to the rubric."

    for key in ALL_KEYS:
        backoff_delay = 5
        for attempt in range(2):
            try:
                temp_client = genai.Client(api_key=key)
                response = temp_client.models.generate_content(
                    model="gemini-3.8-flash",
                    contents=prompt,
                    config={"response_mime_type": "application/json"}
                )
                return json.loads(response.text).get("results", [])
            except Exception as e:
                err_str = str(e).lower()
                if "quota exceeded" in err_str or "daily" in err_str:
                    print(f"❌ Key daily quota exceeded. Switching keys...", flush=True)
                    break
                elif ("503" in err_str or "429" in err_str) and attempt < 1:
                    print(f"⚠️ Rate limited. Waiting {backoff_delay}s before retry...", flush=True)
                    time.sleep(backoff_delay)
                    backoff_delay *= 2
                else:
                    print(f"❌ Key error: {e}. Trying next key...", flush=True)
                    break
    return []

# 3. Gather all answers first
test_runs = []
print("🤖 Generating answers from RAG system...", flush=True)
for idx, item in enumerate(golden_set):
    print(f" [{idx+1}/{len(golden_set)}] Querying: {item['q']}", flush=True)
    try:
        ans = safe_ask_book(item["q"])
    except Exception as e:
        ans = f"ERROR: {e}"

    print(f"   └─ Answered successfully.", flush=True)
    test_runs.append({"q": item["q"], "expected": item["expected"], "actual": ans})
    time.sleep(5)

# 4. Score everything in one batch API request
print("\n⚖️ Submitting answers to the Batch Judge...", flush=True)
scores = judge_all_answers(test_runs)

# 5. Output results
total_score = 0
print("\n--- Evaluation Results ---", flush=True)
for idx, item in enumerate(test_runs):
    score_item = next((s for s in scores if s.get("q_index") == idx), None)
    score = score_item.get("score", 1) if score_item else 1
    reason = score_item.get("reason", "Could not evaluate due to quota") if score_item else "Skipped"

    total_score += score
    print(f"Q: {item['q']}", flush=True)
    print(f"Expected: {item['expected']}", flush=True)
    print(f"Actual:   {item['actual']}", flush=True)
    print(f"Score:    {score}/5 | Reason: {reason}\n", flush=True)

print(f"🏆 Optimized Pipeline Final Score: {total_score}/{len(golden_set) * 5}", flush=True)

🤖 Generating answers from RAG system...
 [1/10] Querying: What is the run time for binary search?
⚠️ Service spike or 429 hit. Retrying with next available key...
⚠️ Service spike or 429 hit. Retrying with next available key...
⚠️ Service spike or 429 hit. Retrying with next available key...
 [2/10] Querying: How does selection sort work?
 [3/10] Querying: What is the difference between an array and a linked list?
 [4/10] Querying: What is a recursive function?
 [5/10] Querying: What is the worst-case run time of quicksort?
 [6/10] Querying: What is a hash table used for?
 [7/10] Querying: What is a directed graph?
 [8/10] Querying: What is Dijkstra's algorithm used for?
 [9/10] Querying: What is a greedy algorithm?
⚠️ Service spike or 429 hit. Retrying with next available key...
 [10/10] Querying: What is the knapsack problem?
⚠️ Service spike or 429 hit. Retrying with next available key...

⚖️ Submitting answers to the Batch Judge...
⚠️ Rate limited. Waiting 5s before retry...

--- E

In [10]:
# Diagnostic cell to check available models under the new google-genai SDK
from google import genai
try:
    client = genai.Client(api_key=ALL_KEYS[0])
    print("Listing active model names:")
    for m in client.models.list():
        print(m.name)
except Exception as e:
    print(f"Error listing models: {e}")

Listing active model names:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/